In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("desafio_04_filtros_ordenacao")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 18:08:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Parte A — Leitura e inspeção

In [3]:
df = spark.read.csv("/workspace/datasets/04_filtros_ordenacao/clientes.csv", header= True, inferSchema= True)

In [4]:
df.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- renda_mensal: double (nullable = true)
 |-- sexo: string (nullable = true)
 |-- ativo: boolean (nullable = true)
 |-- data_cadastro: date (nullable = true)



In [5]:
df.count()

300

In [6]:
df.rdd.getNumPartitions()

1

Por que é importante conhecer o schema antes de aplicar filtros e ordenações, especialmente quando estamos trabalhando com arquivos CSV?

Resposta: É importante conhecer o schema antes de aplicar filtros e ordenações, pois precisamos saber a tipagem e a formatação dos dados, conhecendo o schema é possivel garantir performance, integridade dos dados e otimização de recursos, principalmente em arquivos do tipo CSV onde não possuem metadados nativos como arquivos do tipo parquet. 

Parte B — Filtros com AND

In [25]:
df_filtro_and = (
    df.select("cliente_id", "nome", "idade", "cidade", "renda_mensal")
    .filter((F.col("idade") >= 35) & (F.col("renda_mensal") > 5000))
)

In [26]:
df_filtro_and.count()

148

Esse filtro exige que o Spark redistribua os registros entre partições? Explique relacionando sua resposta a transformações Narrow e Wide.

Resposta: Esse filtro não redistribui os dados entre partições, logo essa é uma transformação narrow.

Parte C — Filtros com OR e AND

In [27]:
df_filtro_and_or = (
    df.select("cliente_id", "nome", "idade", "cidade", "renda_mensal")
    .filter(((F.col("cidade") == "Salvador") | (F.col("cidade") == "Recife")) & (F.col("idade") >= 30))
)

In [28]:
df_filtro_and_or.count()

76

O que pode acontecer quando utilizamos & e | sem organizar corretamente os parênteses das expressões?

Resposta: Caso os parenteses não sejam organizados na forma correta para diferenciar filtros, o spark pode haver uma desorganização a depender do objetivo da consulta

Parte D — Negação de condições

In [14]:
df.count()

300

In [13]:
df_not_salvador = (
    df.filter(~(F.col("cidade") == "Salvador"))
)

In [15]:
df_not_salvador.count()

200

Qual a função do operador ~ no PySpark? Como ele se diferencia dos operadores & e |?

Resposta: O operador ~ no PySpark trazer todas os registros que não fazem jus ao que está no filtro, no exemplo assim, eu solicito ao spark para ele trazer todas as cidades, menos 'Salvador', por tanto o ~ é um operador de negação. Ele se diferencia do & pois essa operação tras registros que atendam as ambas as condições e | retorna registros que atendam uma das condições

Parte E — Ordenação global e Limit

In [30]:
df_limit = (
    df.select("cliente_id", "nome", "cidade", "renda_mensal")
    .orderBy(F.col("renda_mensal").desc())
    .limit(10)
)

In [31]:
df_limit.show()

+----------+-----------+----------------+------------+
|cliente_id|       nome|          cidade|renda_mensal|
+----------+-----------+----------------+------------+
|       212|Cliente 212|Feira de Santana|    14998.33|
|        19| Cliente 19|Feira de Santana|    14974.64|
|       159|Cliente 159|       São Paulo|     14967.5|
|        30| Cliente 30|        Salvador|     14830.7|
|       255|Cliente 255|Feira de Santana|    14822.31|
|       187|Cliente 187|        Salvador|    14726.64|
|       229|Cliente 229|        Salvador|    14719.11|
|        70| Cliente 70|       São Paulo|    14711.11|
|         8|  Cliente 8|       São Paulo|    14631.79|
|       126|Cliente 126|        Salvador|    14613.72|
+----------+-----------+----------------+------------+



1. Por que uma ordenação global pode exigir Shuffle?

Resposta: O shuffle ocorre quando o spark precisar redistribuir dados entre partições, então como o orderby é uma função que tem como objetivo ordernar os dados, se esses dados estiverem em multiplas partições isso pode acabar gerando um shuffle 

2. O limit(10) significa necessariamente que o Spark lê apenas 10 registros?

Resposta: Não, ainda utilizando a função limit o spark ainda sim percorre todas as linhas do dataframe, mas retorna ao usuario final a quantidade de linhas informadas no limit()

3. Por que ordenar um dataset com milhões de linhas pode ser mais caro do que aplicar um filtro simples?

Resposta: Ordenar um dataset com milhões de linhas custuma ser mais caro, pois na grande maioria das vezes acaba gerando um shuffle

Parte F — Análise do Physical Plan

In [32]:
df_filtro_and.explain(True)

== Parsed Logical Plan ==
'Filter (('idade >= 35) AND ('renda_mensal > 5000))
+- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#23]
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double
Filter ((idade#19 >= 35) AND (renda_mensal#23 > cast(5000 as double)))
+- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#23]
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#23]
+- Filter ((isnotnull(idade#19) AND isnotnull(renda_mensal#23)) AND ((idade#19 >= 35) AND (renda_mensal#23 > 5000.0)))
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,

Explique o que o Catalyst fez com as condições utilizadas.

Resposta: O Catalyst tem como objetivo otimizar as funções do spark atuando no Optimized Logical Plan, nesse cenario em especifico, ele otimiza os filtros para que idade seja maior que 35 e a renda mensal maior q 5K, e ambas as colunas não podem ser nulas

In [33]:
df_limit.explain(True)

== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Project [cliente_id#17, nome#18, cidade#20, renda_mensal#23]
         +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, cidade: string, renda_mensal: double
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Project [cliente_id#17, nome#18, cidade#20, renda_mensal#23]
         +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Project [cliente_id#17, nome#18, cidade#20, renda_mensal#23]
         +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo

Por que duas operações aparentemente simples, filter() e orderBy(), podem apresentar custos de execução tão diferentes?

Resposta: O que dita os custos das operações é se elas vão acabar gerando shuffle ou não, evento bem custoso para o spark, no primeiro exemplo df_filtro_and, estamos apenas aplicando filtros especificos em algumas colunas, isso acaba não redistribuindo dados entre partições, não gerando shuffle, cada filtro é realizada em cada partição de forma individual, ja no segundo exemplo df_limit, como estamos utilizando um orderby, essa operação acaba gerando um shuffle, ou seja o spark acaba redistribuindo os dados entre partições para entregar o resultado final

Parte G — Análise final de Engenharia de Dados

Imagine um DataFrame com 50 milhões de registros, distribuídos em 8 Spark Partitions.
Você precisa encontrar os 20 clientes de maior renda mensal.

1. Como o Spark pode executar a filtragem de registros de forma paralela?

Resposta: O spark pode executar a filtragem de forma paralela utilizando partições, como o spark é um motor de processamento distribuido, é possivel distribuir dados entre partições e efetuar filtros com os dados espalhados graças ao paralelismo.

2. O que muda quando precisamos encontrar os maiores valores globalmente?

Resposta: Esse cenario acaba gerando um shuffle, quando precisamos encontra valores globais gera o exchange, e com isso ocorre uma redistribuição de dados entre partições, esse fenomeno acaba sendo custoso para o spark. Mas na abordagem do codigo que foi utilizada nesse desafio, acaba ocorrendo o TakeOrderedAndProject o spark evita enviar os 50 milhões de dados e ordena prmeiro localmente apenas os 20 maiores registros e depois globalmente

3. Qual o papel das Spark Partitions nesse processamento?

Resposta: As partições no spark servem para facilitar o processamento de dados, mas aqui é preciso haver um adendo, mais partições não quer dizer maior otimização, velocidade depende de processamento depende de outros fatores, como memoria, cpu e etc. Mas as spark partitions custumam ser beneficas em cenarios de big data, onde podemos aproveitar o paralelismo do spark para processar os dados.

4. Em quais circunstâncias um Exchange pode aparecer no Physical Plan?

Resposta: Um Exchange pode aparecer no Physical Plan quando há necessidade de redistribuição de dados entre partições, fenomeno conhecido como shuffle.

5. Por que devemos inspecionar explain(True) antes de afirmar que ocorreu um Shuffle?

Resposta: Com o explain(True), conseguimos visualizar todo o planejamento do spark naquele codigo em especifico, para afirmarmos que ocorreu um shuffle em uma operação o Physical Plan informar que ocorreu um exchange, ou seja um sinal verde para o shuffle. No meu exemplo em especifico os dados não foram redistribuidos entre partições pois existe apenas 1 partição, o que ocorreu aqui foi um TakeOrderedAndProject.

Se TakeOrderedAndProject apareceu no seu Physical Plan, por que seria incorreto afirmar automaticamente que o Spark executou uma ordenação global completa com Shuffle?

Resposta: Aparecer um TakeOrderedAndProject no Physical Plan do Spark não significa necessariamente que houve uma ordenação global completa com Shuffle porque esse operador é uma otimização projetada especificamente para evitar o custo de ordenar todo o conjunto de dados.
O Spark utiliza essa estratégia principalmente quando executa operações como orderBy ou limit. O comportamento real depende de onde e como os dados estão distribuídos